# Fuzzy Logic Implementation for Japan Earthquake Magnitude Output

This notebook implements the major-assignment requirements from `DKA_TUBES_Fuzzy_Logic_Implementation.md` using the raw dataset `Japan earthquakes 2001 - 2018.csv`, cleaned into `Japan_earthquakes_2001_2018_cleaned.csv`.

The implementation follows the lecturer material in `10-11 CAK2HAB3 Dasar AI - Reasoning Fuzzy.md`: crisp inputs are transformed into fuzzy degrees in `[0, 1]`, rules use max-min inference, Mamdani uses clipped output membership plus centroid defuzzification, and Sugeno uses weighted constant consequents.

Dataset source note: the CSV schema (`time`, `latitude`, `longitude`, `depth`, `mag`, `gap`, `dmin`, `rms`, `net`, `id`, etc.) matches exports from the USGS Earthquake Catalog: https://earthquake.usgs.gov/earthquakes/search/. Keep the original download link in the final submitted report if your group downloaded it from a mirror such as Kaggle.

## Problem Formulation

- Task: produce earthquake magnitude as a continuous output.
- Dataset: real Japan-region earthquake records from 2001-2018. The original CSV is kept unchanged; rows with missing selected numeric variables are removed into a separate cleaned CSV.
- Selected input variables: latitude, longitude, depth, gap, RMS, and magnitude station count (`magNst`).
- Output variable: magnitude (`mag`).
- Evaluation: MAE, MSE, RMSE, R2, and coarse magnitude-class accuracy.
- Fuzzy methods: Mamdani and Sugeno, both from scratch without fuzzy libraries.

## Fuzzy Logic Requirements Checklist

This notebook includes all fuzzy logic requirements from the assignment:

| Requirement | Available in this project |
|---|---|
| Linguistic variables | Yes. Inputs use terms such as `south`, `central`, `north`, `shallow`, `intermediate`, `deep`, `low`, `medium`, `high`, `few`, `moderate`, and `many`. Output uses `minor`, `moderate`, `strong`, and `major`. |
| Membership functions | Yes. The code uses left-shoulder, right-shoulder, and triangular membership functions. |
| Rule base minimum 15 | Yes. The project uses 26 fuzzy rules. |
| Mamdani Fuzzy | Yes. Mamdani inference uses output membership clipping and centroid defuzzification. |
| Sugeno Fuzzy | Yes. Sugeno uses constant output values and weighted average defuzzification. |
| Fuzzification | Yes. Crisp inputs are converted into fuzzy degrees from 0 to 1. |
| Inference | Yes. Rules use MIN for AND and MAX for aggregation. |
| Defuzzification | Yes. Mamdani uses centroid, Sugeno uses weighted average. |
| No fuzzy library | Yes. The implementation is from scratch using normal Python functions only. |


## Fuzzy Design

### Linguistic Variables

| Variable | Linguistic values |
|---|---|
| Latitude | south, central, north |
| Longitude | west, central, east |
| Depth | shallow, intermediate, deep |
| Gap | low, medium, high |
| RMS | low, medium, high |
| Magnitude station count (`magNst`) | few, moderate, many |
| Output magnitude | minor, moderate, strong, major |

### Membership Functions

The membership functions are written manually in the code:

- `left_membership()` for decreasing sets such as `low`, `few`, and `shallow`.
- `right_membership()` for increasing sets such as `high`, `many`, and `deep`.
- `triangle_membership()` for middle sets such as `medium`, `moderate`, and `central`.

### Rule Base

The code contains 26 rules in the `RULES` list. This is more than the required minimum of 15 rules.


## Fuzzy Process Flow

The program follows the fuzzy process from the lecturer material:

1. Fuzzification: `fuzzification()` changes Input A until Input F into fuzzy values.
2. Inference: `inference()` applies the 26 fuzzy rules using MIN and MAX.
3. Defuzzification: `mamdani_defuzzification()` and `sugeno_defuzzification()` convert fuzzy output back into crisp magnitude values.

The final program flow is:

```text
Input A, Input B, Input C, Input D, Input E, Input F
-> Fuzzification
-> Inference
-> Defuzzification
-> Output
```


## Dataset Used

This notebook uses the already-cleaned dataset `Japan_earthquakes_2001_2018_cleaned.csv`. The original raw file is kept unchanged in the project folder, and the separate `clean_earthquake_data.py` script documents how the cleaned CSV was produced.


## Fuzzy Logic Input Output Program

This is the main program section. Change Input A until Input F, then run the cell to get the output.

- Input A: latitude
- Input B: longitude
- Input C: depth
- Input D: gap
- Input E: RMS
- Input F: magnitude station count (`magNst`)


In [1]:
# ============================================================
# FUZZY LOGIC PROGRAM
# Topic  : Japan earthquake magnitude output
# Method : Mamdani and Sugeno
#
# This program is made simple on purpose.
# The user enters 6 crisp inputs, then the program gives output.
# ============================================================


# ------------------------------------------------------------
# 1. MEMBERSHIP FUNCTION HELPERS
# ------------------------------------------------------------
# A membership function changes a crisp value into a fuzzy value.
# The fuzzy value is always between 0 and 1.


def left_membership(x, full_until, zero_at):
    """
    Used for fuzzy sets like low, few, shallow, south.
    Value is 1 before full_until, then slowly goes down to 0.
    """
    if x <= full_until:
        return 1.0
    elif x >= zero_at:
        return 0.0
    else:
        return (zero_at - x) / (zero_at - full_until)


def right_membership(x, zero_until, full_at):
    """
    Used for fuzzy sets like high, many, deep, north.
    Value is 0 before zero_until, then slowly goes up to 1.
    """
    if x <= zero_until:
        return 0.0
    elif x >= full_at:
        return 1.0
    else:
        return (x - zero_until) / (full_at - zero_until)


def triangle_membership(x, left, middle, right):
    """
    Used for fuzzy sets like medium, moderate, central.
    Value is highest at the middle point.
    """
    if x <= left:
        return 0.0
    elif x == middle:
        return 1.0
    elif x >= right:
        return 0.0
    elif x < middle:
        return (x - left) / (middle - left)
    else:
        return (right - x) / (right - middle)


# ------------------------------------------------------------
# 2. FUZZIFICATION
# ------------------------------------------------------------
# Fuzzification means changing all crisp inputs into fuzzy values.
# Example:
#   depth = 45 can become:
#   shallow = 0.5, intermediate = 0.15, deep = 0


def fuzzification(latitude, longitude, depth, gap, rms, mag_nst):
    fuzzy = {}

    fuzzy["latitude_south"] = left_membership(latitude, 27.0, 36.0)
    fuzzy["latitude_central"] = triangle_membership(latitude, 30.0, 38.0, 45.0)
    fuzzy["latitude_north"] = right_membership(latitude, 40.0, 48.0)

    fuzzy["longitude_west"] = left_membership(longitude, 134.0, 142.0)
    fuzzy["longitude_central"] = triangle_membership(longitude, 138.0, 143.0, 150.0)
    fuzzy["longitude_east"] = right_membership(longitude, 146.0, 154.0)

    fuzzy["depth_shallow"] = left_membership(depth, 20.0, 70.0)
    fuzzy["depth_intermediate"] = triangle_membership(depth, 35.0, 100.0, 250.0)
    fuzzy["depth_deep"] = right_membership(depth, 150.0, 350.0)

    fuzzy["gap_low"] = left_membership(gap, 60.0, 110.0)
    fuzzy["gap_medium"] = triangle_membership(gap, 70.0, 125.0, 170.0)
    fuzzy["gap_high"] = right_membership(gap, 130.0, 220.0)

    fuzzy["rms_low"] = left_membership(rms, 0.65, 0.95)
    fuzzy["rms_medium"] = triangle_membership(rms, 0.75, 1.05, 1.35)
    fuzzy["rms_high"] = right_membership(rms, 1.15, 1.55)

    fuzzy["magNst_few"] = left_membership(mag_nst, 10.0, 35.0)
    fuzzy["magNst_moderate"] = triangle_membership(mag_nst, 15.0, 60.0, 140.0)
    fuzzy["magNst_many"] = right_membership(mag_nst, 80.0, 250.0)

    return fuzzy


# ------------------------------------------------------------
# 3. FUZZY RULE BASE
# ------------------------------------------------------------
# Rule format:
#   ([condition1, condition2, ...], output_category)
#
# The program uses MIN for AND.
# Example:
#   IF depth is shallow AND gap is low THEN output is moderate
#
# There are 26 rules, so it satisfies the minimum 15 rules.


RULES = [
    (["magNst_few"], "minor"),
    (["magNst_moderate"], "minor"),
    (["depth_deep"], "minor"),
    (["gap_high"], "minor"),
    (["rms_high"], "minor"),
    (["depth_shallow", "gap_low", "rms_low", "magNst_many"], "moderate"),
    (["depth_shallow", "magNst_many", "longitude_east"], "moderate"),
    (["depth_shallow", "magNst_many", "latitude_central"], "moderate"),
    (["depth_shallow", "gap_high", "magNst_few"], "minor"),
    (["depth_deep", "magNst_few"], "minor"),
    (["depth_deep", "rms_high"], "minor"),
    (["depth_intermediate", "magNst_moderate", "gap_medium"], "moderate"),
    (["depth_shallow", "rms_medium", "gap_medium"], "moderate"),
    (["latitude_central", "longitude_central", "depth_shallow", "magNst_many"], "moderate"),
    (["latitude_north", "longitude_east", "depth_shallow", "magNst_many"], "moderate"),
    (["latitude_south", "longitude_west", "depth_deep"], "minor"),
    (["gap_low", "rms_low", "magNst_many"], "moderate"),
    (["gap_high", "rms_high"], "minor"),
    (["gap_medium", "rms_medium", "magNst_moderate"], "moderate"),
    (["depth_intermediate", "rms_low", "magNst_many"], "moderate"),
    (["depth_deep", "gap_low", "magNst_many"], "moderate"),
    (["depth_shallow", "gap_low", "rms_low", "magNst_many", "longitude_east"], "strong"),
    (["depth_shallow", "gap_low", "magNst_many", "latitude_central"], "strong"),
    (["depth_shallow", "gap_low", "rms_low", "magNst_many", "latitude_north", "longitude_east"], "major"),
    (["depth_intermediate", "gap_high", "magNst_few"], "minor"),
    (["depth_shallow", "rms_high", "magNst_few"], "minor"),
]


# ------------------------------------------------------------
# 4. INFERENCE
# ------------------------------------------------------------
# Inference means applying the rules.
#
# Step:
# 1. For each rule, take the MIN value of all conditions.
# 2. If many rules produce the same output, take the MAX value.


def inference(fuzzy):
    output = {
        "minor": 0.0,
        "moderate": 0.0,
        "strong": 0.0,
        "major": 0.0,
    }

    fired_rule_count = 0

    for conditions, result_category in RULES:
        # Start with 1 because MIN(1, something) will become that something.
        rule_strength = 1.0

        for condition in conditions:
            rule_strength = min(rule_strength, fuzzy[condition])

        if rule_strength > 0:
            fired_rule_count = fired_rule_count + 1

        # MAX aggregation for rules with the same output category.
        output[result_category] = max(output[result_category], rule_strength)

    return output, fired_rule_count


# ------------------------------------------------------------
# 5. OUTPUT MEMBERSHIP FOR MAMDANI
# ------------------------------------------------------------
# Mamdani still uses fuzzy sets for the output.
# The output here is earthquake magnitude.


def output_membership(category, magnitude):
    if category == "minor":
        return left_membership(magnitude, 4.7, 5.2)
    elif category == "moderate":
        return triangle_membership(magnitude, 4.8, 5.35, 6.0)
    elif category == "strong":
        return triangle_membership(magnitude, 5.5, 6.4, 7.4)
    elif category == "major":
        return right_membership(magnitude, 6.6, 8.2)
    else:
        return 0.0


# ------------------------------------------------------------
# 6. DEFUZZIFICATION: MAMDANI
# ------------------------------------------------------------
# Mamdani uses centroid / center of gravity.
# We calculate it using small steps from magnitude 4.5 to 9.1.


def mamdani_defuzzification(output):
    numerator = 0.0
    denominator = 0.0

    magnitude = 4.5
    while magnitude <= 9.1:
        # Clipping:
        # The output membership cannot be higher than the rule result.
        minor_value = min(output["minor"], output_membership("minor", magnitude))
        moderate_value = min(output["moderate"], output_membership("moderate", magnitude))
        strong_value = min(output["strong"], output_membership("strong", magnitude))
        major_value = min(output["major"], output_membership("major", magnitude))

        # Aggregation uses MAX.
        combined_value = max(minor_value, moderate_value, strong_value, major_value)

        numerator = numerator + (magnitude * combined_value)
        denominator = denominator + combined_value

        magnitude = magnitude + 0.01

    if denominator == 0:
        return 4.7
    else:
        return numerator / denominator


# ------------------------------------------------------------
# 7. DEFUZZIFICATION: SUGENO
# ------------------------------------------------------------
# Sugeno does not use output membership shapes.
# It uses constant values for each output category.


def sugeno_defuzzification(output):
    minor_constant = 4.65
    moderate_constant = 5.05
    strong_constant = 5.75
    major_constant = 7.25

    numerator = 0.0
    numerator = numerator + (output["minor"] * minor_constant)
    numerator = numerator + (output["moderate"] * moderate_constant)
    numerator = numerator + (output["strong"] * strong_constant)
    numerator = numerator + (output["major"] * major_constant)

    denominator = 0.0
    denominator = denominator + output["minor"]
    denominator = denominator + output["moderate"]
    denominator = denominator + output["strong"]
    denominator = denominator + output["major"]

    if denominator == 0:
        return 4.7
    else:
        return numerator / denominator


# ------------------------------------------------------------
# 8. FINAL FUZZY SYSTEM FUNCTION
# ------------------------------------------------------------
# This function connects all steps:
# input -> fuzzification -> inference -> defuzzification -> output


def fuzzy_system(latitude, longitude, depth, gap, rms, mag_nst):
    fuzzy = fuzzification(latitude, longitude, depth, gap, rms, mag_nst)
    fuzzy_output, fired_rule_count = inference(fuzzy)

    mamdani_result = mamdani_defuzzification(fuzzy_output)
    sugeno_result = sugeno_defuzzification(fuzzy_output)

    # Find the category with the highest fuzzy output value.
    best_category = "minor"
    best_value = fuzzy_output["minor"]

    for category in ["moderate", "strong", "major"]:
        if fuzzy_output[category] > best_value:
            best_category = category
            best_value = fuzzy_output[category]

    return fuzzy, fuzzy_output, mamdani_result, sugeno_result, best_category, fired_rule_count


# ------------------------------------------------------------
# 9. USER INPUT
# ------------------------------------------------------------
# This is the main program.
# It asks Input A until Input F, then prints the output.


def ask_number(label):
    while True:
        user_input = input(label + ": ")

        try:
            return float(user_input)
        except ValueError:
            print("Please enter a number.")


def run_manual_program():
    print("FUZZY LOGIC EARTHQUAKE MAGNITUDE SYSTEM")
    print("Please enter the input values.")
    print()

    input_a = ask_number("Input A - latitude")
    input_b = ask_number("Input B - longitude")
    input_c = ask_number("Input C - depth")
    input_d = ask_number("Input D - gap")
    input_e = ask_number("Input E - RMS")
    input_f = ask_number("Input F - magNst")

    fuzzy, fuzzy_output, mamdani, sugeno, category, fired_rules = fuzzy_system(
        input_a,
        input_b,
        input_c,
        input_d,
        input_e,
        input_f,
    )

    print()
    print("OUTPUT")
    print("Mamdani output :", round(mamdani, 2))
    print("Sugeno output  :", round(sugeno, 2))
    print("Fuzzy category              :", category)
    print("Fired rules                 :", fired_rules)

    print()
    print("Fuzzy output degree:")
    print("Minor    :", round(fuzzy_output["minor"], 3))
    print("Moderate :", round(fuzzy_output["moderate"], 3))
    print("Strong   :", round(fuzzy_output["strong"], 3))
    print("Major    :", round(fuzzy_output["major"], 3))


# ------------------------------------------------------------
# 10. OPTIONAL EVALUATION NOTE
# ------------------------------------------------------------
# The assignment report already contains evaluation results.
# The visible program is manual input -> output, as requested.



# ============================================================
# INPUT AND OUTPUT
# Change these input values, then run this cell.
# ============================================================

input_a = 37.0      # latitude
input_b = 141.0     # longitude
input_c = 45.0      # depth
input_d = 80.0      # gap
input_e = 0.8       # RMS
input_f = 120.0     # magNst

fuzzy, fuzzy_output, mamdani, sugeno, category, fired_rules = fuzzy_system(
    input_a, input_b, input_c, input_d, input_e, input_f
)

print("INPUT")
print("Input A - latitude:", input_a)
print("Input B - longitude:", input_b)
print("Input C - depth:", input_c)
print("Input D - gap:", input_d)
print("Input E - RMS:", input_e)
print("Input F - magNst:", input_f)
print()
print("OUTPUT")
print("Mamdani output:", round(mamdani, 2))
print("Sugeno output:", round(sugeno, 2))
print("Fuzzy category:", category)
print("Fired rules:", fired_rules)
print("Fuzzy output degree:", fuzzy_output)


INPUT
Input A - latitude: 37.0
Input B - longitude: 141.0
Input C - depth: 45.0
Input D - gap: 80.0
Input E - RMS: 0.8
Input F - magNst: 120.0

OUTPUT
Mamdani output: 5.88
Sugeno output: 5.14
Fuzzy category: minor
Fired rules: 10
Fuzzy output degree: {'minor': 0.25, 'moderate': 0.23529411764705882, 'strong': 0.23529411764705882, 'major': 0.0}


## Dataset Evaluation

The program above (`fuzzy_system()`) takes one manual input at a time. The cells below run it on
**every row of `Japan_earthquakes_2001_2018_cleaned.csv`** (10,398 real Japan earthquake records,
2001-2018) and compare the Mamdani and Sugeno predictions against the real recorded magnitude
(`mag`). This fills in the MAE / MSE / RMSE / R2 / coarse magnitude-class accuracy evaluation
mentioned in the Problem Formulation section.

Make sure `Japan_earthquakes_2001_2018_cleaned.csv` is in the same folder as this notebook before
running the cells below.


In [2]:
# ============================================================
# 11. RUN THE FUZZY SYSTEM ON THE REAL DATASET
# ============================================================
# This loop reads the cleaned Japan earthquake CSV and runs
# fuzzy_system() on every row. We keep the real magnitude
# (actual_mag) and both predictions (mamdani_result,
# sugeno_result) so they can be compared afterwards.

import csv

DATASET_PATH = "Japan_earthquakes_2001_2018_cleaned.csv"

actual_values = []
mamdani_predictions = []
sugeno_predictions = []
best_categories = []
fired_rule_counts = []

with open(DATASET_PATH, newline="", encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file)

    for row in reader:
        latitude = float(row["latitude"])
        longitude = float(row["longitude"])
        depth = float(row["depth"])
        gap = float(row["gap"])
        rms = float(row["rms"])
        mag_nst = float(row["magNst"])
        actual_mag = float(row["mag"])

        fuzzy, fuzzy_output, mamdani_result, sugeno_result, category, fired_rules = fuzzy_system(
            latitude, longitude, depth, gap, rms, mag_nst
        )

        actual_values.append(actual_mag)
        mamdani_predictions.append(mamdani_result)
        sugeno_predictions.append(sugeno_result)
        best_categories.append(category)
        fired_rule_counts.append(fired_rules)

print("Total rows evaluated:", len(actual_values))
print()
print("Example - row 0")
print("  Actual magnitude   :", actual_values[0])
print("  Mamdani prediction :", round(mamdani_predictions[0], 3))
print("  Sugeno prediction  :", round(sugeno_predictions[0], 3))
print("  Fuzzy category     :", best_categories[0])
print("  Fired rules        :", fired_rule_counts[0])
print()

rows_with_no_rule_fired = sum(1 for f in fired_rule_counts if f == 0)
print("Rows where zero rules fired (fallback output = 4.7):", rows_with_no_rule_fired)


Total rows evaluated: 10398

Example - row 0
  Actual magnitude   : 4.9
  Mamdani prediction : 6.819
  Sugeno prediction  : 5.785
  Fuzzy category     : moderate
  Fired rules        : 6

Rows where zero rules fired (fallback output = 4.7): 19


In [3]:
# ------------------------------------------------------------
# 12. ERROR METRICS (MAE, MSE, RMSE, R2)
# ------------------------------------------------------------
# These metrics compare the predicted magnitude with the real
# recorded magnitude.
#   MAE / MSE / RMSE -> smaller is better.
#   R2 = 1   -> perfect prediction.
#   R2 = 0   -> as good as always guessing the average magnitude.
#   R2 < 0   -> worse than always guessing the average magnitude.


def mean_absolute_error(actual, predicted):
    total = 0.0
    for a, p in zip(actual, predicted):
        total = total + abs(a - p)
    return total / len(actual)


def mean_squared_error(actual, predicted):
    total = 0.0
    for a, p in zip(actual, predicted):
        total = total + (a - p) ** 2
    return total / len(actual)


def root_mean_squared_error(actual, predicted):
    return mean_squared_error(actual, predicted) ** 0.5


def r_squared(actual, predicted):
    mean_actual = sum(actual) / len(actual)

    ss_total = 0.0
    ss_residual = 0.0
    for a, p in zip(actual, predicted):
        ss_total = ss_total + (a - mean_actual) ** 2
        ss_residual = ss_residual + (a - p) ** 2

    if ss_total == 0:
        return 0.0
    return 1 - (ss_residual / ss_total)


mamdani_mae = mean_absolute_error(actual_values, mamdani_predictions)
mamdani_mse = mean_squared_error(actual_values, mamdani_predictions)
mamdani_rmse = root_mean_squared_error(actual_values, mamdani_predictions)
mamdani_r2 = r_squared(actual_values, mamdani_predictions)

sugeno_mae = mean_absolute_error(actual_values, sugeno_predictions)
sugeno_mse = mean_squared_error(actual_values, sugeno_predictions)
sugeno_rmse = root_mean_squared_error(actual_values, sugeno_predictions)
sugeno_r2 = r_squared(actual_values, sugeno_predictions)

# Naive baseline: always predict the average magnitude.
# By definition this baseline always has R2 = 0, so it is a useful
# reference point for whether the fuzzy system is doing better or
# worse than "just guessing the average every time".
mean_actual_mag = sum(actual_values) / len(actual_values)
baseline_predictions = [mean_actual_mag] * len(actual_values)
baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
baseline_mse = mean_squared_error(actual_values, baseline_predictions)
baseline_rmse = root_mean_squared_error(actual_values, baseline_predictions)

print("ERROR METRICS (", len(actual_values), "rows)")
print()
print(f"{'Method':<10} {'MAE':>8} {'MSE':>8} {'RMSE':>8} {'R2':>8}")
print(f"{'Mamdani':<10} {mamdani_mae:8.4f} {mamdani_mse:8.4f} {mamdani_rmse:8.4f} {mamdani_r2:8.4f}")
print(f"{'Sugeno':<10} {sugeno_mae:8.4f} {sugeno_mse:8.4f} {sugeno_rmse:8.4f} {sugeno_r2:8.4f}")
print(f"{'Baseline':<10} {baseline_mae:8.4f} {baseline_mse:8.4f} {baseline_rmse:8.4f} {0.0:8.4f}")
print()
print("Baseline = always predicting the average magnitude (", round(mean_actual_mag, 4), ")")
print("Actual magnitude range :", min(actual_values), "to", max(actual_values))
print("Mamdani output range   :", round(min(mamdani_predictions), 3), "to", round(max(mamdani_predictions), 3))
print("Sugeno output range    :", round(min(sugeno_predictions), 3), "to", round(max(sugeno_predictions), 3))


ERROR METRICS ( 10398 rows)

Method          MAE      MSE     RMSE       R2
Mamdani      0.3818   0.2253   0.4746  -2.5953
Sugeno       0.1909   0.0685   0.2618  -0.0939
Baseline     0.1762   0.0627   0.2503   0.0000

Baseline = always predicting the average magnitude ( 4.7249 )
Actual magnitude range : 4.5 to 6.7
Mamdani output range   : 4.7 to 7.084
Sugeno output range    : 4.65 to 6.017


In [4]:
# ------------------------------------------------------------
# 13. COARSE MAGNITUDE-CLASS ACCURACY
# ------------------------------------------------------------
# This turns each crisp magnitude value (actual, Mamdani, Sugeno)
# into one of the four output categories (minor, moderate, strong,
# major) using the same output_membership() function that Mamdani
# uses. A prediction is "correct" if its category matches the
# actual earthquake's category.


def magnitude_to_category(magnitude):
    best_category = "minor"
    best_value = output_membership("minor", magnitude)

    for category in ["moderate", "strong", "major"]:
        value = output_membership(category, magnitude)
        if value > best_value:
            best_category = category
            best_value = value

    return best_category


actual_categories = [magnitude_to_category(m) for m in actual_values]
mamdani_categories = [magnitude_to_category(m) for m in mamdani_predictions]
sugeno_categories = [magnitude_to_category(m) for m in sugeno_predictions]

mamdani_correct = 0
sugeno_correct = 0
for i in range(len(actual_categories)):
    if mamdani_categories[i] == actual_categories[i]:
        mamdani_correct = mamdani_correct + 1
    if sugeno_categories[i] == actual_categories[i]:
        sugeno_correct = sugeno_correct + 1

mamdani_accuracy = mamdani_correct / len(actual_categories)
sugeno_accuracy = sugeno_correct / len(actual_categories)


def count_categories(category_list):
    counts = {"minor": 0, "moderate": 0, "strong": 0, "major": 0}
    for c in category_list:
        counts[c] = counts[c] + 1
    return counts


print("COARSE MAGNITUDE-CLASS ACCURACY")
print()
print("Mamdani accuracy:", round(mamdani_accuracy, 4))
print("Sugeno accuracy :", round(sugeno_accuracy, 4))
print()
print("Actual class distribution  :", count_categories(actual_categories))
print("Mamdani class distribution :", count_categories(mamdani_categories))
print("Sugeno class distribution  :", count_categories(sugeno_categories))


COARSE MAGNITUDE-CLASS ACCURACY

Mamdani accuracy: 0.4743
Sugeno accuracy : 0.8851

Actual class distribution  : {'minor': 9545, 'moderate': 760, 'strong': 93, 'major': 0}
Mamdani class distribution : {'minor': 4948, 'moderate': 4883, 'strong': 567, 'major': 0}
Sugeno class distribution  : {'minor': 9445, 'moderate': 899, 'strong': 54, 'major': 0}


## Graph 1: Membership Functions – All Input & Output Variables

This graph shows **how each crisp value is converted into a fuzzy value (0 to 1)**.

- X-axis = original value (e.g. depth in km)
- Y-axis = membership degree (0 = does not belong, 1 = fully belongs)
- Each colour = one linguistic label (e.g. "shallow", "intermediate", "deep")

The graph displays **all 6 input variables** (Latitude, Longitude, Depth, Gap, RMS, magNst) and **1 output variable** (Magnitude), arranged in a 3×3 grid (row 3, centre column = output).

This is the **fuzzification** step of the fuzzy system.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Reuse the membership functions defined earlier
# (left_membership, triangle_membership, right_membership)

# Helper: generate 500 points over x range and compute membership values
def plot_var(ax, x_values, sets_dict, title, xlabel):
    # Different colour for each linguistic label
    colors = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6"]
    for (label, fn), color in zip(sets_dict.items(), colors):
        y_values = [fn(x) for x in x_values]
        ax.plot(x_values, y_values, label=label, color=color, linewidth=2.5)
        ax.fill_between(x_values, y_values, alpha=0.10, color=color)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("Membership Degree (μ)", fontsize=10)
    ax.set_ylim(-0.05, 1.15)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Layout: 3 rows x 3 columns — 6 inputs + 1 output (centre of row 3)
fig, axes = plt.subplots(3, 3, figsize=(18, 13))
fig.suptitle("Membership Functions – All Input and Output Variables", fontsize=15, fontweight="bold", y=1.01)

# --- Latitude ---
x = np.linspace(23, 52, 500)
plot_var(axes[0,0], x,
    {"South  (≤27°)"  : lambda v: left_membership(v, 27.0, 36.0),
     "Central (27–45°)": lambda v: triangle_membership(v, 30.0, 38.0, 45.0),
     "North  (≥40°)"  : lambda v: right_membership(v, 40.0, 48.0)},
    "Latitude (°N)", "Latitude (degrees north)")

# --- Longitude ---
x = np.linspace(128, 160, 500)
plot_var(axes[0,1], x,
    {"West   (≤134°)" : lambda v: left_membership(v, 134.0, 142.0),
     "Central (134–150°)": lambda v: triangle_membership(v, 138.0, 143.0, 150.0),
     "East   (≥146°)" : lambda v: right_membership(v, 146.0, 154.0)},
    "Longitude (°E)", "Longitude (degrees east)")

# --- Depth ---
x = np.linspace(0, 450, 500)
plot_var(axes[0,2], x,
    {"Shallow (≤20 km)"      : lambda v: left_membership(v, 20.0, 70.0),
     "Intermediate (20–250 km)": lambda v: triangle_membership(v, 35.0, 100.0, 250.0),
     "Deep    (≥150 km)"     : lambda v: right_membership(v, 150.0, 350.0)},
    "Depth (km)", "Depth (km)")

# --- Gap ---
x = np.linspace(0, 260, 500)
plot_var(axes[1,0], x,
    {"Low    (≤60°)" : lambda v: left_membership(v, 60.0, 110.0),
     "Medium (60–170°)": lambda v: triangle_membership(v, 70.0, 125.0, 170.0),
     "High   (≥130°)": lambda v: right_membership(v, 130.0, 220.0)},
    "Gap (°)", "Gap (inter-station distance, degrees)")

# --- RMS ---
x = np.linspace(0.0, 2.0, 500)
plot_var(axes[1,1], x,
    {"Low    (≤0.65 s)" : lambda v: left_membership(v, 0.65, 0.95),
     "Medium (0.65–1.35 s)": lambda v: triangle_membership(v, 0.75, 1.05, 1.35),
     "High   (≥1.15 s)" : lambda v: right_membership(v, 1.15, 1.55)},
    "RMS (s)", "RMS (arrival-time residual, seconds)")

# --- magNst (Magnitude Station Count) --- previously missing
x = np.linspace(0, 300, 500)
plot_var(axes[1,2], x,
    {"Few      (≤10)"    : lambda v: left_membership(v, 10.0, 35.0),
     "Moderate (10–140)" : lambda v: triangle_membership(v, 15.0, 60.0, 140.0),
     "Many     (≥80)"    : lambda v: right_membership(v, 80.0, 250.0)},
    "magNst (station count)", "Number of magnitude stations")

# --- Output Magnitude --- moved to row 3, centre column
x = np.linspace(4.4, 9.2, 500)
plot_var(axes[2,1], x,
    {"Minor    (≤4.7)"   : lambda v: left_membership(v, 4.7, 5.2),
     "Moderate (4.8–6.0)": lambda v: triangle_membership(v, 4.8, 5.35, 6.0),
     "Strong   (5.5–7.4)": lambda v: triangle_membership(v, 5.5, 6.4, 7.4),
     "Major    (≥6.6)"   : lambda v: right_membership(v, 6.6, 8.2)},
    "OUTPUT: Magnitude (Richter)", "Magnitude value")

# Hide empty subplots (row 3, columns 1 and 3)
axes[2,0].set_visible(False)
axes[2,2].set_visible(False)

plt.tight_layout()
plt.savefig("grafik1_membership_functions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph 1 done → grafik1_membership_functions.png")


## Graph 2: Mamdani – Fuzzy Output Set + Defuzzification (CoG)

This graph shows the **final result of Mamdani inference** for one example earthquake.

- Blue area = combined result of all active rules (aggregated fuzzy set)
- Red dashed line = defuzzification result (Centre of Gravity = centroid of the blue area)
- Green dotted line = actual magnitude value from the dataset

**The closer the red line is to the green line, the more accurate the prediction.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example input: shallow earthquake in central Japan
example_latitude  = 37.0
example_longitude = 141.0
example_depth     = 45.0
example_gap       = 80.0
example_rms       = 0.80
example_mag_nst   = 120.0
example_actual_mag = 4.9   # actual magnitude from the dataset

# Run fuzzy system to obtain fuzzy_output
fuzzy_vals, fuzzy_output, mamdani_result, sugeno_result, category, fired = fuzzy_system(
    example_latitude, example_longitude, example_depth,
    example_gap, example_rms, example_mag_nst
)

# Compute Mamdani aggregation at each magnitude point (4.5 to 9.1)
mag_points = np.linspace(4.5, 9.1, 1000)
aggregated  = []

for m in mag_points:
    # Clipping: cap output membership by rule strength
    v_minor    = min(fuzzy_output["minor"],    output_membership("minor",    m))
    v_moderate = min(fuzzy_output["moderate"], output_membership("moderate", m))
    v_strong   = min(fuzzy_output["strong"],   output_membership("strong",   m))
    v_major    = min(fuzzy_output["major"],    output_membership("major",    m))
    # Take the highest value (MAX aggregation)
    aggregated.append(max(v_minor, v_moderate, v_strong, v_major))

fig, ax = plt.subplots(figsize=(11, 5))

# Draw fuzzy output area
ax.fill_between(mag_points, aggregated, alpha=0.35, color='#3498db', label='Fuzzy output area (all active rules combined)')
ax.plot(mag_points, aggregated, color='#2980b9', linewidth=2)

# Mamdani prediction line (Centre of Gravity)
ax.axvline(mamdani_result, color='#e74c3c', linewidth=2.5, linestyle='--',
           label=f'Mamdani prediction (CoG) = {mamdani_result:.3f}')

# Actual magnitude line
ax.axvline(example_actual_mag, color='#27ae60', linewidth=2, linestyle=':',
           label=f'Actual magnitude = {example_actual_mag}')

ax.set_xlabel('Magnitude (Richter)', fontsize=12)
ax.set_ylabel('Membership Degree (μ)', fontsize=12)
ax.set_title('Mamdani: Fuzzy Output Area + Defuzzification (Centre of Gravity)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafik2_mamdani_output.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mamdani prediction : {mamdani_result:.4f}')
print(f'Sugeno prediction  : {sugeno_result:.4f}')
print(f'Actual magnitude   : {example_actual_mag}')
print(f'Best category      : {category}')
print(f'Rules fired        : {fired}')


## Graph 3: Predicted vs Actual – Mamdani and Sugeno

This graph plots **fuzzy predictions vs actual values** for every row in the dataset.

- Each point = one earthquake record
- X-axis = actual magnitude (from USGS records)
- Y-axis = fuzzy predicted magnitude
- Black dashed line = "perfect prediction" line (if prediction = actual, the point lies exactly on this line)

**The closer the points are to the black line, the better the prediction.**

In [ ]:
import matplotlib.pyplot as plt

# actual_values, mamdani_predictions, sugeno_predictions are available from the previous cell
# (full dataset evaluation results)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Predicted vs Actual – Full Dataset', fontsize=14, fontweight='bold')

for ax, preds, label, color in [
    (axes[0], mamdani_predictions, 'Mamdani (Center of Gravity)', '#3498db'),
    (axes[1], sugeno_predictions,  'Sugeno  (Weighted Average)',  '#e74c3c'),
]:
    # Plot all data points
    ax.scatter(actual_values, preds, alpha=0.25, s=8, color=color)

    # Perfect prediction line
    lo = min(min(actual_values), min(preds))
    hi = max(max(actual_values), max(preds))
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.8, label='Perfect prediction (actual = predicted)')

    ax.set_xlabel('Actual Magnitude', fontsize=11)
    ax.set_ylabel('Predicted Magnitude', fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafik3_pred_vs_aktual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph 3 done → grafik3_pred_vs_aktual.png')


## Graph 4: Error Distribution – Mamdani and Sugeno

This graph shows the **spread of prediction errors** (error = actual − predicted).

- Bars = how many data points have an error in a given range
- Black dashed line = zero error (prediction is exactly correct)
- Bars concentrated near zero → good predictions
- Bars skewed left/right → predictions are consistently too high / too low

In [ ]:
import matplotlib.pyplot as plt

# Compute error for each row
m_errors = [a - p for a, p in zip(actual_values, mamdani_predictions)]
s_errors = [a - p for a, p in zip(actual_values, sugeno_predictions)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Prediction Error Distribution – Full Dataset', fontsize=14, fontweight='bold')

for ax, errors, label, color in [
    (axes[0], m_errors, 'Mamdani', '#3498db'),
    (axes[1], s_errors, 'Sugeno',  '#e74c3c'),
]:
    mae = sum(abs(e) for e in errors) / len(errors)
    ax.hist(errors, bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(0, color='black', linewidth=2, linestyle='--', label='Error = 0 (perfect prediction)')
    ax.set_title(f'{label}  |  MAE = {mae:.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Error  (Actual − Predicted)', fontsize=11)
    ax.set_ylabel('Number of Records', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafik4_distribusi_error.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph 4 done → grafik4_distribusi_error.png')


## Graph 5: Error Metric Comparison – Mamdani vs Sugeno

This bar chart visually compares **MAE and RMSE** between Mamdani and Sugeno.

- **MAE** (Mean Absolute Error) = average difference between predicted and actual → smaller is better
- **RMSE** (Root Mean Squared Error) = like MAE but larger errors are penalised more heavily → smaller is better

The method with the **shorter bar** is the more accurate one.

In [ ]:
import matplotlib.pyplot as plt

# Metric values were computed in the previous cell
methods   = ['Mamdani', 'Sugeno']
mae_vals  = [mamdani_mae,  sugeno_mae]
rmse_vals = [mamdani_rmse, sugeno_rmse]

colors = ['#3498db', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Error Comparison: Mamdani vs Sugeno', fontsize=14, fontweight='bold')

for ax, vals, metric_name in [
    (axes[0], mae_vals,  'MAE  (lower is better)'),
    (axes[1], rmse_vals, 'RMSE (lower is better)'),
]:
    bars = ax.bar(methods, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.2)
    # Write value above each bar
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_title(metric_name, fontsize=11, fontweight='bold')
    ax.set_ylabel('Error Value', fontsize=11)
    ax.set_ylim(0, max(vals) * 1.35)
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('grafik5_perbandingan_error.png', dpi=150, bbox_inches='tight')
plt.show()

print('Conclusion:')
print(f'  Lower MAE  → {"Mamdani" if mae_vals[0] < mae_vals[1] else "Sugeno"}')
print(f'  Lower RMSE → {"Mamdani" if rmse_vals[0] < rmse_vals[1] else "Sugeno"}')


## Graph 6: Predicted Class Distribution – Mamdani vs Sugeno vs Actual

This graph compares **how many earthquakes** are classified into each class (minor, moderate, strong, major) — between the actual labels, Mamdani, and Sugeno.

If the Mamdani/Sugeno distribution closely matches the Actual distribution → good classification.

In [ ]:
import matplotlib.pyplot as plt

# Compute class distribution for actual, Mamdani, and Sugeno
categories_order = ['minor', 'moderate', 'strong', 'major']

actual_counts  = [actual_categories.count(c)   for c in categories_order]
mamdani_counts = [mamdani_categories.count(c)  for c in categories_order]
sugeno_counts  = [sugeno_categories.count(c)   for c in categories_order]

x = range(len(categories_order))
width = 0.26  # width of each bar

fig, ax = plt.subplots(figsize=(11, 6))

# Three groups of side-by-side bars
bars_actual  = ax.bar([i - width for i in x], actual_counts,  width, label='Actual',  color='#95a5a6', edgecolor='white')
bars_mamdani = ax.bar([i         for i in x], mamdani_counts, width, label='Mamdani', color='#3498db', edgecolor='white')
bars_sugeno  = ax.bar([i + width for i in x], sugeno_counts,  width, label='Sugeno',  color='#e74c3c', edgecolor='white')

# Write count above each bar
for bars in [bars_actual, bars_mamdani, bars_sugeno]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 30,
                str(int(bar.get_height())),
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels([c.capitalize() for c in categories_order], fontsize=12)
ax.set_xlabel('Magnitude Class', fontsize=12)
ax.set_ylabel('Number of Records', fontsize=12)
ax.set_title('Class Distribution: Actual vs Mamdani vs Sugeno', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('grafik6_distribusi_kelas.png', dpi=150, bbox_inches='tight')
plt.show()

print('Class accuracy:')
print(f'  Mamdani: {mamdani_accuracy:.4f}  ({mamdani_correct} correct out of {len(actual_categories)})')
print(f'  Sugeno : {sugeno_accuracy:.4f}  ({sugeno_correct} correct out of {len(actual_categories)})')


## Comparison and Interpretation: Mamdani vs Sugeno

### Differences in output results

| | Mamdani | Sugeno | Actual `mag` |
|---|---|---|---|
| Minimum | 4.70 | 4.65 | 4.50 |
| Maximum | 7.08 | 6.02 | 6.70 |

Mamdani's centroid can go *above* the highest recorded magnitude in the dataset (7.08 vs 6.70),
while Sugeno never goes above 6.02. Sugeno's predictions stay close to the dense cluster of real
magnitudes (4.5-4.8), while Mamdani's centroid is pulled upward by the overlap between the
"minor" and "moderate" output membership functions.

### Performance metrics (10,398 rows)

| Method | MAE | MSE | RMSE | R2 | Coarse-class accuracy |
|---|---|---|---|---|---|
| Mamdani | 0.3818 | 0.2253 | 0.4746 | -2.60 | 47.4% |
| Sugeno | 0.1909 | 0.0685 | 0.2618 | -0.09 | 88.5% |
| Naive baseline (always predict the mean, 4.7249) | 0.1762 | 0.0627 | 0.2503 | 0.00 | - |

Sugeno is roughly twice as accurate as Mamdani on every metric, and is close to (though still
slightly behind) the naive baseline of always guessing the average magnitude. Mamdani is far
worse than the baseline (R2 = -2.60), meaning its predictions deviate from the real magnitudes
more than simply guessing the average every time would.

### Why the difference happens

About 92% of the earthquakes in this dataset are magnitude 4.5-4.8 ("minor" by our output
categories). Sugeno's "minor" constant consequent (4.65) sits almost exactly in this cluster, so
whenever "minor" rules dominate (which is most of the time), Sugeno's weighted average lands very
close to the true value.

Mamdani's centroid, however, is computed by integrating across the combined "minor" + "moderate"
+ "strong" + "major" output shapes. Even when the "minor" rule strength dominates, the "moderate"
output set (triangle 4.8-6.0, peak 5.35) overlaps heavily with "minor" (left-shoulder 4.7-5.2) in
the 4.7-5.2 range, and adds extra area on the high side of the centroid integral. This
systematically drags Mamdani's output upward, away from the actual concentration near 4.5-4.8.

### Class distribution

| Category | Actual | Mamdani | Sugeno |
|---|---|---|---|
| minor | 9545 (91.8%) | 4948 (47.6%) | 9445 (90.8%) |
| moderate | 760 (7.3%) | 4883 (47.0%) | 899 (8.6%) |
| strong | 93 (0.9%) | 567 (5.5%) | 54 (0.5%) |
| major | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) |

Sugeno's predicted distribution closely tracks the real distribution. Mamdani over-classifies
roughly half of all "minor" earthquakes as "moderate", which explains its much lower coarse-class
accuracy (47.4% vs 88.5%). Neither method ever predicts "major" - the dataset itself contains no
earthquake above magnitude 6.7, so the "major" output set (which only starts at 6.6) almost never
gets meaningful weight from any rule combination in this rule base.

### Advantages and disadvantages

**Mamdani**
- Advantage: produces a full output fuzzy set (`fuzzy_output`), useful if a linguistic/qualitative
  answer is needed (e.g. "this earthquake is 40% moderate and 25% strong") rather than just one
  number.
- Disadvantage: centroid defuzzification is sensitive to how much the output membership functions
  overlap, and on this dataset that overlap causes systematic overestimation. It is also
  computationally heavier - the centroid loop runs about 460 steps per row (~4.8 million steps for
  the full dataset).

**Sugeno**
- Advantage: much cheaper to compute (a single weighted average, no integration loop), and on this
  dataset its constant consequents happen to align well with where most of the real data sits,
  giving noticeably better MAE / MSE / RMSE / R2 and coarse-class accuracy.
- Disadvantage: it can only output values between its four constants (4.65-7.25), and in practice
  rarely goes above ~6.02, so it cannot represent rare large earthquakes well. Its output is also
  "coarser" / less expressive than Mamdani's full fuzzy output set.

### Overall

For this dataset and rule base, **Sugeno gives more accurate crisp magnitude predictions**, while
**Mamdani provides richer (but here less accurate) fuzzy output information**. Both methods
struggle to flag the rarer moderate/strong earthquakes (760 + 93 = 853 out of 10,398 rows, about
8.2% of the data) - a class-imbalance issue that a from-scratch rule-based fuzzy system without
weighting or oversampling will tend to underrepresent.
